## Imports

In [65]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
)
from sklearn.model_selection import (
    FixedThresholdClassifier,
    GroupShuffleSplit,
    GridSearchCV,
    GroupKFold,
)
from sklearn.pipeline import Pipeline

### Load full dataset

In [66]:
source = pd.read_json("python_dataset.jsonl", lines=True)

### Prepare Dataset

In [67]:
df = source

# df = df.head(10000)

df = df.drop(columns="docstring")

# Convert df from wide format to long format. Originally it was idx, human_code, gpt_code, qwen_code ...
# But i need seperate rows per code snippet. With melt I transform it to idx, code, source
df = pd.melt(
    df,
    id_vars="hm_index",
    value_vars=["human_code", "chatgpt_code", "dsc_code", "qwen_code"],
    var_name="source",
    value_name="code",
)

df = df.dropna(axis=0)

# Add new column thats 0 or 1 depending on source. Human-written is 0, AI-generated is 1
df["ai_generated"] = df["source"].ne("human_code").astype(int)

df = df.rename(columns={"hm_index": "task_id"})

# Simplify task_id values and sort df
df["task_id"] = df.groupby("task_id").ngroup() + 1
df = df.sort_values(by=["task_id"])

# Move colum order so source comes last
df = df[["task_id", "code", "ai_generated", "source"]]

### Split df into training and test

In [ ]:
# Split code into input X and ouput y
all_task_ids = df["task_id"]
X = df["code"]
y = df["ai_generated"]

# The df has multiple rows per coding task. e.g. task 1 has 4 rows (human-, gpt-, qwen- and dsc- code snippets)
# with GroupShuffleShift I split the the rows but dont put rows with the same task_id into train AND test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_val_idx, test_idx = next(gss_test.split(X, y, groups=all_task_ids))
test_X, test_y = X.iloc[test_idx], y.iloc[test_idx]


# train val split
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

train_val_df = df.iloc[train_val_idx]
X_tv = train_val_df["code"]
y_tv = train_val_df["ai_generated"]
train_val_task_ids = train_val_df["task_id"]

train_idx, val_idx = next(gss_val.split(X_tv, y_tv, groups=train_val_task_ids))

train_X, train_y = X_tv.iloc[train_idx], y_tv.iloc[train_idx]
val_X, val_y = X_tv.iloc[val_idx], y_tv.iloc[val_idx]


ValueError: Found input variables with inconsistent numbers of samples: [912796, 912796, 1140996]

### Build Pipeline and optimize it for f1_macro score

In [91]:
# Init a Pipeline with 3 steps:
# 1. Count vectorizer to make a count vector for each row in the provided data (only the code)
# 2. TfidfTransformer takes count vectors from previous step and produces tf-idf vectors
# 3. Logreg as our model

pipeline = Pipeline(
    steps=[
        ("cv", CountVectorizer()),
        ("tfidf", TfidfTransformer()),
        ("logreg", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

param_grid = {"logreg__C": [0.1, 1, 10, 100]}

group_k_fold = GroupKFold(n_splits=5)

grid_search_cv = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    refit=True,
    cv=group_k_fold,
    scoring="f1_macro",
    verbose=2,
    n_jobs=-1,
)

grid_search_cv.fit(train_X, train_y, groups=train_idx)

best_pipeline = grid_search_cv.best_estimator_

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV] END ......................................logreg__C=0.1; total time=  49.8s
[CV] END ......................................logreg__C=0.1; total time=  49.5s
[CV] END ......................................logreg__C=0.1; total time=  44.7s
[CV] END ......................................logreg__C=0.1; total time=  59.0s
[CV] END ......................................logreg__C=0.1; total time=  59.8s
[CV] END ........................................logreg__C=1; total time=  59.6s
[CV] END ........................................logreg__C=1; total time= 1.1min
[CV] END ........................................logreg__C=1; total time= 1.1min
[CV] END ........................................logreg__C=1; total time= 1.1min
[CV] END ........................................logreg__C=1; total time= 1.2min
[CV] END .......................................logreg__C=10; total time= 1.1min
[CV] END .......................................l

### Treshold tuning using the validation set

In [ ]:
pred_proba = best_pipeline.predict_proba(val_X)

positive_class_probs = pred_proba[:, 1]

fpr, tpr, threshold = roc_curve(val_y, positive_class_probs)

for f, t, thr in zip(fpr, tpr, threshold):
    print(f"Threshhold: {thr:.3f} -> FPR: {f:.3f} -> TPR: {t:.3f}")

# threshold 0.856 for fpr 5%, tpr 0.778

Threshhold: inf -> FPR: 0.000 -> TPR: 0.000
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.000
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.000
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.000
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.000
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.001
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.001
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.001
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.001
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.001
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.001
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.002
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.002
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.002
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.002
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.002
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.002
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.002
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.002
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.002
Threshhold: 1.000 -> FPR: 0.000 -> TPR: 0.002
Threshhold: 1.000 -> FPR: 0.000 -> T